# COVID FakeNews impact on Event-Based-Surveillance Systems

In [1]:
%load_ext autoreload
%autoreload 2

## Libraries

### Installing

In [2]:
#!pip install pandas
#!pip install tqdm
#!pip install nltk
#!pip install gatenlp
#!pip install py4j
#!pip install pyodide
#!pip install ipywidgets
#!pip install neo4j

### Standard

In [23]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
import glob
import re

## Globals

In [5]:
path_to_fakenews_csv = "/data/dataRapide/gabriel/git/DDPF/data/covid/fakenews_journal/documents/euvsdisinfo_full.csv"

## FakeNews Dataset Preparation

The fakenews dataset can be recreated by two sets of publicly available datasets:
* Trustworthy positives : Aylien COVID dataset: https://aylien.com/resources/datasets/coronavirus-dataset
* Fakenews : By retrieving the FakeNews COVID dataset **csv** from https://github.com/JAugusto97/euvsdisinfo

You can extract from Aylien a folder of texts from November 2019 in one folder. It is needed to do the same with the csv from the fakenews dataset. 

### Extracting reports from the csv

In [6]:
fakenewsdf = pd.read_csv(path_to_fakenews_csv)

In [26]:
covid_kw = [
    "Coronavirus",
    "Vaccination",
    "Biological weapons",
    "Chemical weapons/attack",
    "Conspiracy theory",
    "Laboratory",
    "Virus / bacteria threat"
]
# Convert the covid_kw list to lowercase
covid_kw = [kw.lower() for kw in covid_kw]

In [53]:
# Function to check if any covid_kw keyword is in the 'keywords' column
def contains_covid_kw(keywords):
    # Split the 'keywords' string into a list, convert to lowercase, and check intersection
    keywords_list = [kw.strip().lower() for kw in str(keywords).split(',')]
    return any(kw in keywords_list for kw in covid_kw)


# Filter the DataFrame using the contains_covid_kw function
filtered_df = fakenewsdf[fakenewsdf['keywords'].apply(contains_covid_kw)]

# Step 1: Filter by 'class' being 'disinformation'
filtered_df = filtered_df[filtered_df['class'] == 'disinformation']

# Step 2: Convert 'published_date' to datetime format (assuming 'day-month-year' format)
filtered_df['published_date'] = pd.to_datetime(filtered_df['published_date'], format='%d-%m-%Y', errors='coerce')

# Step 3: Filter for rows in November 2019
filtered_df = filtered_df[
    (filtered_df['published_date'].dt.month.isin([11, 12])) & 
    (filtered_df['published_date'].dt.year == 2019)
]
# Step 4: Filter by 'article_language' being 'English'
filtered_df = filtered_df[filtered_df['article_language'] == 'English']

In [54]:
filtered_df

,debunk_id,debunk_title,keywords,article_id,article_title,article_publisher,article_domain,article_url,article_text,article_language,debunk_date,published_date,class
35,fff2b6b7-8a3b-4013-87cb-8f31b505b786,Disinfo: Zionist money already corrupting the ...,"Anti-Semitism, Manipulated elections/referendu...",0a763b2c-aaff-470f-9cbc-eb628874df77,Zionist Money Already Corrupting the 2020 Elec...,russia-insider.com,russia-insider.com,https://russia-insider.com/en/zionist-money-al...,"Watching the last Democratic debate, you would...",English,12-12-2019,2019-12-12,disinformation
247,4a06c6bd-da5e-4a89-bc08-a7e602940872,Disinfo: Anglo-Saxons are staging a colour rev...,"Catholic church, Colour revolutions, Conspirac...",a6c61293-6853-4a26-ad51-675877f1cd8d,Троянский конь англосаксов,stoletie.ru,stoletie.ru,http://www.stoletie.ru/politika/trojanskij_kon...,Что делают украинские боевики на улицах мятежн...,Russian,11-12-2019,2019-12-10,disinformation
312,02a63ab4-6d73-447d-bb4c-d5ba22c97170,"Disinfo: WADA's doping accusations, just like ...","GRU, Sergei Skripal, West, Anti-Russian, WADA,...",a8aa1b96-9902-4b1d-92c9-021c8e3e3def,WADA: торжество абсурда или заговор?,stoletie.ru,stoletie.ru,http://www.stoletie.ru/tekuschiiy_moment/wada_...,За что решили наказать российских спортсменов ...,Russian,11-12-2019,2019-12-10,disinformation
313,cda1848c-c4c6-411c-81c1-860fc1a4f02a,Disinfo: Nazi took over Ukraine with a coup,"Eastern Ukraine, Ukrainian disintegration, Con...",02c0abdb-7c09-4f3b-b8bf-d0779457f4f3,هل يتعين على روسيا “التدخل” في الانتخابات المق...,syriafriends.net,syriafriends.net,https://syriafriends.net/2019/12/%D9%87%D9%84-...,لدى اليهود ما يسمى بالـ “خوبتسا” أو ما يمكن تس...,Arabic,20-12-2019,2019-12-21,disinformation
314,cda1848c-c4c6-411c-81c1-860fc1a4f02a,Disinfo: Nazi took over Ukraine with a coup,"Eastern Ukraine, Ukrainian disintegration, Con...",8d0dee4e-8852-43c3-83b2-774072336908,هَلْ يَتَعَيَّنُ عَلَى رُوسِيَا التَّدَخُّلِ ف...,kachaf.com,kachaf.com,https://www.kachaf.com/details.php?n=5dfcd40ed...,"لدى اليهود ما يسمى بالـ ""خوبتسا"" أو ما يمكن تس...",Arabic,20-12-2019,2019-12-20,disinformation
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5493,b05dbe28-1516-4eab-b5ef-3dc1d14e0734,Disinfo: Crimea referendum prevented USA plans...,"Military, Nuclear issues, Crimea, Barack Obama...",eea43a34-4c94-4149-9f71-b805aad3695a,Amerika pravila planove da preuzme kontrolu na...,vesti,vesti-online.com,https://www.vesti-online.com/amerika-pravila-p...,Sjedinjene Američke Države su nameravale da pr...,Serbian,04-11-2019,2019-11-04,disinformation
5494,b05dbe28-1516-4eab-b5ef-3dc1d14e0734,Disinfo: Crimea referendum prevented USA plans...,"Military, Nuclear issues, Crimea, Barack Obama...",df04fb39-759f-404c-8a6f-cf2802bcb694,Otkriven zastrašujući Obamin plan: Vašington h...,sd.rs,srbijadanas.com,https://www.srbijadanas.com/vesti/svet/otkrive...,"""Amerikan herald tribjun"" piše da je strategij...",Croatian,04-11-2019,2019-11-04,disinformation
9215,486382a6-36de-4df9-bc12-3d5055d9ebc5,Disinfo: The White Helmets are preparing a new...,"White Helmets, Syrian War, Provocation, false ...",52e8c068-4559-41c6-ae97-8a568f8f88ef,"زاخاروفا: ""الخوذ البيضاء"" تنسق مع الإرهابيين ل...",إذاعة النور,alnour.com.lb,https://www.alnour.com.lb/news/politics/392553...,حذرت المتحدثة باسم وزارة الخارجية الروسية ماري...,Arabic,21-05-2021,2019-11-01,disinformation
9221,486382a6-36de-4df9-bc12-3d5055d9ebc5,Disinfo: The White Helmets are preparing a new...,"White Helmets, Syrian War, Provocation, false ...",265dc266-b709-4e8e-bcff-e16589640d3a,موسكو: إرهابيو “الخوذ البيضاء” يجهزون عمليات ا...,albaathmedia.sy,albaathmedia.sy,http://albaathmedia.sy/2019/11/01/%D9%85%D9%88...,حذرت المتحدثة باسم وزارة الخارجية الروسية ماري...,Arabic,21-05-2021,2019-11-01,disinformation


In [20]:
for x in a:
    if "virus / bacteria threat" in str(x).lower():
        print(x)

Biological weapons, Virus / bacteria threat, laboratory, health, Conspiracy theory
Virus / bacteria threat, Climate
Biological weapons, laboratory, Virus / bacteria threat, Conspiracy theory
Lugar Laboratory, Virus / bacteria threat, laboratory, Biological weapons, Conspiracy theory
coronavirus, Virus / bacteria threat, Biological weapons, Conspiracy theory
Biological weapons, laboratory, Virus / bacteria threat, Drugs, War in Ukraine, Conspiracy theory, Invasion of Ukraine
coronavirus, Virus / bacteria threat, Bill Gates, Biological weapons, Conspiracy theory
coronavirus, Virus / bacteria threat, Russian superiority, European values
Lugar Laboratory, Virus / bacteria threat, Biological weapons
Virus / bacteria threat, Ukrainian disintegration, Ukrainian statehood
coronavirus, Lugar Laboratory, Virus / bacteria threat, laboratory, Biological weapons
Virus / bacteria threat, Biological weapons, Anti-Russian, Ukrainian statehood
coronavirus, Virus / bacteria threat, fake news, Biological